# Grid intraday de reação de mercado — Payroll

Gera **apenas a figura** (grid 3×3 no estilo do Chart Grid da Bloomberg) da reação
intraday à divulgação do payroll (Nonfarm Payrolls), exibida inline. **Não** exporta
Word, **não** atualiza xlsx e **não** salva PNG — só o gráfico na tela.

Reusa o core do FOMC (`reports.fomc.core`): mesmos 9 painéis (Nasdaq 100, S&P 500,
Russell 2000, UST 2Y, UST 10Y, 2s10s, OIS Fwd 1Y1Y, DXY, VIX), mudando apenas o
evento marcado — a divulgação às 8:30 ET — e o estilo (`style="bloomberg"`: área
preenchida, eixo Y à direita com caixa do último valor, subtítulo
`T=valor  variação  %  HH:MM`).

**Cada linha termina no último tick de fato**, não na borda do gráfico: o eixo X vai
até `GRID_X_END_BRT` (área em branco à direita) e o horário no subtítulo de cada
painel confirma até quando há dado. Rodando numa data já encerrada, o Bloomberg
devolve a sessão inteira — aí a linha vai até o fechamento, como esperado.

Os 9 painéis compartilham a **mesma janela e os mesmos ticks** no eixo X, para
comparação direta. Por isso o VIX tem um vazio à esquerda: ele não tem cotação
overnight (começa ~04:15 BRT, contra ~21:00 do dia anterior nos futuros).

1. Configure a data e o horário da divulgação (fixo em ET, convertido p/ Brasília).
2. Rode a última célula para buscar os dados no Bloomberg e desenhar o grid.

In [ ]:
# Setup + imports
from datetime import datetime
from pathlib import Path

from reports._paths import INPUT, OUTPUT

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Reuso do core do FOMC: fetch intraday + grid 3x3 (painéis idênticos)
from reports.fomc.core.data_loader import fetch_market_reaction, _TZ_ET, _TZ_BRT
from reports.fomc.core.word_export import create_market_reaction_grid

print("Módulos carregados.")

In [ ]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================
# Data da divulgação do payroll (YYYYMMDD) — em geral primeira sexta do mês,
# mas o BLS antecipa quando cai em feriado (conferir: bls.gov/schedule).
# Precisa ser um dia com pregão já ocorrido (data futura/feriado -> RuntimeError
# "No intraday data returned from Bloomberg").
# ATENÇÃO: numa data já encerrada o Bloomberg devolve a sessão inteira, então as
# linhas vão até o fim do pregão. Rodando no próprio dia, cada painel termina no
# último tick de fato (o subtítulo "T=... HH:MM" mostra esse horário).
RELEASE_DATE = '20260904'

# Horário da divulgação — FIXO em horário de NY (ET); conversão p/ Brasília é
# automática (auto-DST via zoneinfo — 8:30 ET = 9:30 BRT no verão de NY,
# 10:30 BRT no inverno).
PAYROLL_RELEASE_TIME_ET = "08:30:00"

# Eixo X se estende até este horário (em Brasília), mesmo sem dados até lá.
GRID_X_END_BRT = "16:00:00"

# Estilo do grid: "bloomberg" reproduz a estrutura do Chart Grid (área
# preenchida, eixo Y à direita com caixa do último valor, subtítulo com variação
# e horário do último tick) na paleta clara. "word" = visual do relatório FOMC.
GRID_STYLE = "bloomberg"

# Evento marcado com linha vertical no grid: (chave, rótulo, estilo de linha)
PAYROLL_EVENTS = [("payroll", "Payroll", "-")]

In [ ]:
# Gera e exibe o grid (sem salvar arquivo)
def _et_to_brt(release_date: str, hms: str) -> pd.Timestamp:
    """'HH:MM:SS' em ET (na data da divulgação) -> Timestamp em Brasília."""
    rd = datetime.strptime(release_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(rd, t), tz=_TZ_ET).tz_convert(_TZ_BRT)

def _brt_time(release_date: str, hms: str) -> pd.Timestamp:
    """'HH:MM:SS' já em Brasília -> Timestamp na data da divulgação."""
    rd = datetime.strptime(release_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(rd, t), tz=_TZ_BRT)

event_times = {"payroll": _et_to_brt(RELEASE_DATE, PAYROLL_RELEASE_TIME_ET)}
axis_end = _brt_time(RELEASE_DATE, GRID_X_END_BRT)
print(f"Payroll: {event_times['payroll']:%H:%M} BRT  |  eixo X até {axis_end:%H:%M} BRT")

market_data = fetch_market_reaction(RELEASE_DATE)

# Até que horário cada painel realmente tem dados (a linha termina aí, não na
# borda do gráfico) — útil para conferir a defasagem entre painéis.
for _col in market_data.columns:
    _s = market_data[_col].dropna()
    if not _s.empty:
        print(f"  {_col:<14} último tick {_s.index[-1]:%H:%M} BRT  ({len(_s)} pontos)")

fig = create_market_reaction_grid(
    market_data,
    event_times=event_times,
    meeting_date=RELEASE_DATE,
    axis_end=axis_end,
    events=PAYROLL_EVENTS,
    style=GRID_STYLE,
)  # sem output_path -> não salva PNG
display(fig)
plt.close(fig)  # evita exibição duplicada pelo backend inline